In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
#importing all the essential libraries
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from tensorflow.keras.layers import Input, Dense, Flatten, ReLU, Conv2D, DepthwiseConv2D, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras import Model

#Here in our dataset we Dont have any explict Labelling so,ImageDAtaGenerator will make the folder name as its label
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from keras.callbacks import ReduceLROnPlateau #This is to Modify the Learning Rate according to the performance of the model


2024-03-03 07:57:41.834073: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-03 07:57:41.834191: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-03 07:57:41.982196: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
import os
from sklearn.model_selection import train_test_split

# Define the path to your dataset folder
dataset_path = '/kaggle/input/anthracnose-disease-in-chilli-mobile-captured/Anthracnose_in_chilli_Mobile_captured'

# Get the list of all image files in the dataset folder
image_files = []
labels = []
for class_name in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, class_name)
    if os.path.isdir(class_path):
        for image_file in os.listdir(class_path):
            image_files.append(os.path.join(class_path, image_file))
            labels.append(class_name)

# Split the dataset into training and validation sets
train_files, valid_files, train_labels, valid_labels = train_test_split(image_files, labels, test_size=0.2, random_state=42)

# Now you have train_files, valid_files, train_labels, and valid_labels containing the paths and labels for your training and validation sets


In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# Define data augmentation parameters
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest')

# Define your data generators for training and validation
train_data = datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=True)

validation_data = datagen.flow_from_dataframe(
    dataframe=valid_df,
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=True)


NameError: name 'train_df' is not defined

In [ ]:
mobilenet_tf = keras.Sequential([
    keras.layers.Conv2D(64, (5,5), activation=tf.nn.relu,input_shape=(224, 224, 3)),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(),

    keras.layers.Conv2D(64, (3,3), activation=tf.nn.relu,padding = 'Same'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Conv2D(128, (3,3), activation=tf.nn.relu,padding = 'Same'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Flatten(),

    keras.layers.Dense(256, activation=tf.nn.relu),
    keras.layers.Dense(1, activation = tf.nn.sigmoid)
])

In [ ]:
mobilenet_tf.summary()

In [ ]:
callbacks=ReduceLROnPlateau(monitor='accuracy', 
                              factor=0.2, 
                              patience=2, 
                              mode='min', 
                              min_lr=1e-6)


In [ ]:
# from keras.optimizers import Adam
from tensorflow.keras.optimizers import Adam
opt = Adam(learning_rate=0.001)
mobilenet_tf.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])


history=mobilenet_tf.fit(train_data,validation_data=validation_data,epochs=30,verbose=1,callbacks=[callbacks])

In [ ]:
history_df=pd.DataFrame(history.history)
history_df.loc[:,['loss','val_loss']].plot()
print("The minimum validation Loss={}".format(history_df['val_loss'].min()))

In [ ]:
history_df=pd.DataFrame(history.history)
history_df.loc[:,['accuracy','val_accuracy']].plot()
print("The minimum validation Accuracy={}".format(history_df['val_accuracy'].min()))

In [ ]:
mobilenet_tf.save('Anthracnose_model.hdf5')

In [ ]:
from tensorflow.keras.preprocessing.image import load_img
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.models import load_model

# Load the entire model
mobilenet_tf = load_model("/kaggle/working/Anthracnose_model.hdf5")

# Load the image using TensorFlow's load_img
img = load_img("/kaggle/input/anthracnose-disease-in-chilli-mobile-captured/Anthracnose_in_chilli_Mobile_captured/Healthy/IMG-20231104-WA0028.jpg", target_size=(224, 224))
img = np.asarray(img)
plt.imshow(img)
img = np.expand_dims(img, axis=0)

# Comment out the following line, as the model is already compiled during training
# mobilenet_tf.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

# Predict
output = mobilenet_tf.predict(img)
threshold = 0.5  # Adjust this threshold as needed
if output[0][0] < threshold:
    print("Anthracnose")
else:
    print("Healthy")

In [ ]:
from tensorflow.keras.preprocessing.image import load_img
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.models import load_model

# Load the entire model
mobilenet_tf = load_model("/kaggle/working/Anthracnose_model.hdf5")

# Load the image using TensorFlow's load_img
img = load_img("/kaggle/input/anthracnose-disease-in-chilli-mobile-captured/Anthracnose_in_chilli_Mobile_captured/Anthracnose/IMG-20231104-WA0137.jpg", target_size=(224, 224))
img = np.asarray(img)
plt.imshow(img)
img = np.expand_dims(img, axis=0)

# Comment out the following line, as the model is already compiled during training
# mobilenet_tf.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

# Predict
output = mobilenet_tf.predict(img)
threshold = 0.5  # Adjust this threshold as needed
if output[0][0] < threshold:
    print("Anthracnose")
else:
    print("Healthy")